In [6]:
import json
from sklearn.model_selection import train_test_split
from collections import Counter

def create_nested_splits_hardcoded_path(filtered_annotation_file, output_file='final_nested_splits.json'):
    """
    Create stratified 8:1:1 splits with nested structure and hardcoded image_path
    """
    
    # Fixed hardcoded path (not used for any logic, just included in output)
    HARDCODED_PATH = "/home/woody/iwi5/iwi5204h/HistGen/Data/WSI/PatchResult/patches/pt_files/dinov2_vitl/PIT_01_02729_01.pt"
    
    # Load your filtered annotations (8352 images)
    with open(filtered_annotation_file, 'r') as f:
        annotations = json.load(f)
    
    print(f"Loaded {len(annotations)} annotations from filtered file")
    
    ids = []
    reports = []
    strat_keys = []
    
    for ann in annotations:
        ids.append(ann['id'])
        reports.append(ann['report'])
        
        # Create stratification key from organ + diagnosis
        report = ann['report']
        organ = report.split(',')[0].strip() if ',' in report else 'Unknown'
        
        diagnosis = 'Unknown_Diagnosis'
        if ';' in report:
            parts = report.split(';', 1)
            if len(parts) > 1:
                diagnosis = parts[1].strip()
        else:
            lines = report.split('\n')
            if len(lines) > 1:
                diagnosis = lines[1].strip()
        
        strat_key = f"{organ}|{diagnosis[:50]}"
        strat_keys.append(strat_key)
    
    # Filter out single-instance categories
    strat_counter = Counter(strat_keys)
    filtered_ids = []
    filtered_reports = []
    filtered_strat = []
    excluded_count = 0
    
    for i, strat_val in enumerate(strat_keys):
        if strat_counter[strat_val] >= 2:
            filtered_ids.append(ids[i])
            filtered_reports.append(reports[i])
            filtered_strat.append(strat_val)
        else:
            excluded_count += 1
    
    if excluded_count > 0:
        print(f"Excluded {excluded_count} samples with single-instance categories")
    
    print(f"Using {len(filtered_ids)} samples for stratified splitting")
    
    # Stratified splits
    train_val_ids, test_ids, train_val_reports, test_reports, train_val_strat, test_strat = train_test_split(
        filtered_ids, filtered_reports, filtered_strat,
        test_size=0.10,
        stratify=filtered_strat,
        random_state=42
    )
    
    train_ids, val_ids, train_reports, val_reports, train_strat, val_strat = train_test_split(
        train_val_ids, train_val_reports, train_val_strat,
        test_size=0.111,
        stratify=train_val_strat,
        random_state=42
    )
    
    # Create NESTED structure with hardcoded path
    nested_splits = {
        "train": [],
        "val": [],
        "test": []
    }
    
    # Add train samples
    for i, id_val in enumerate(train_ids):
        nested_splits["train"].append({
            "id": id_val,
            "report": train_reports[i],
            "image_path": [HARDCODED_PATH],  # Same hardcoded path for all
            "split": "train"
        })
    
    # Add validation samples
    for i, id_val in enumerate(val_ids):
        nested_splits["val"].append({
            "id": id_val,
            "report": val_reports[i],
            "image_path": [HARDCODED_PATH],  # Same hardcoded path for all
            "split": "val"
        })
    
    # Add test samples
    for i, id_val in enumerate(test_ids):
        nested_splits["test"].append({
            "id": id_val,
            "report": test_reports[i],
            "image_path": [HARDCODED_PATH],  # Same hardcoded path for all
            "split": "test"
        })
    
    # Save to JSON file
    with open(output_file, 'w') as f:
        json.dump(nested_splits, f, indent=2)
    
    # Print summary
    print(f"\n=== FINAL NESTED SPLITS SUMMARY ===")
    print(f"Train: {len(train_ids):,} samples")
    print(f"Val:   {len(val_ids):,} samples") 
    print(f"Test:  {len(test_ids):,} samples")
    print(f"Total: {len(filtered_ids):,} samples")
    print(f"\nNested splits with hardcoded paths saved to: {output_file}")
    
    return nested_splits

# Usage
if __name__ == "__main__":
    # UPDATE THIS PATH TO YOUR FILTERED JSON FILE
    filtered_annotation_file = 'filtered_captions.json'
    
    splits = create_nested_splits_hardcoded_path(filtered_annotation_file)


Loaded 8352 annotations from filtered file
Excluded 106 samples with single-instance categories
Using 8246 samples for stratified splitting

=== FINAL NESTED SPLITS SUMMARY ===
Train: 6,597 samples
Val:   824 samples
Test:  825 samples
Total: 8,246 samples

Nested splits with hardcoded paths saved to: final_nested_splits.json


Below is the final one use for splitting. It tahes into account the organ name, the procedure and the diagnosis into account. If that combination 

IF frequency = 1: → Training set
IF frequency = 2: → Training set (both)
IF frequency = 3-4: → 2 to training, 1 to test, 1 to validation
IF frequency ≥ 5: → Apply 8:1:1 stratification


In [1]:
import json
from collections import defaultdict
import random
import os

def extract_info(report):
    """
    Extract organ, procedure, and diagnosis from report.
    Report format: "Organ, procedure;\n diagnosis"
    """
    try:
        header, diagnosis = report.split(';\n', 1)
    except ValueError:
        # Fallback for reports without ;\n separator
        parts = report.split(';')
        if len(parts) > 1:
            header = parts[0]
            diagnosis = ';'.join(parts[1:])
        else:
            header = report
            diagnosis = ""

    # Extract organ and procedure from header
    header_parts = header.split(',', 1)
    organ = header_parts[0].strip() if len(header_parts) > 0 else ""
    procedure = header_parts[1].strip() if len(header_parts) > 1 else ""
    diagnosis = diagnosis.strip()
    
    return organ, procedure, diagnosis

def create_output_dict(item, split, base_path):
    """Create output dictionary with required fields"""
    id_ = item['id']
    # Replace .tiff with .pt for image_path
    pt_name = id_.replace('.tiff', '.pt')
    image_path = [base_path + pt_name]
    
    return {
        'id': id_,
        'report': item['report'],
        'image_path': image_path,
        'split': split
    }

def stratified_split(input_file, output_file):
    """
    Perform stratified splitting based on organ, procedure, and diagnosis
    """
    # Constants
    BASE_IMAGE_PATH = "/home/woody/iwi5/iwi5204h/HistGen/Data/WSI/PatchResult/patches/pt_files/dinov2_vitl/"
    
    # Load input data
    with open(input_file, 'r') as f:
        data = json.load(f)
    
    print(f"Loaded {len(data)} samples from {input_file}")
    
    # Group by organ, procedure, diagnosis combination
    groups = defaultdict(list)
    
    for item in data:
        organ, procedure, diagnosis = extract_info(item['report'])
        key = (organ, procedure, diagnosis)
        groups[key].append(item)
    
    print(f"Found {len(groups)} unique organ-procedure-diagnosis combinations")
    
    # Initialize splits
    splits = {'train': [], 'val': [], 'test': []}
    
    # Set random seed for reproducibility
    random.seed(42)
    
    # Statistics tracking
    stats = {'singleton': 0, 'double': 0, 'small': 0, 'normal': 0}
    
    # Apply splitting rules
    for key, items in groups.items():
        n = len(items)
        random.shuffle(items)  # Randomize order within group
        
        if n == 1 or n == 2:
            # All go to training
            stats['singleton' if n == 1 else 'double'] += 1
            for item in items:
                splits['train'].append(create_output_dict(item, 'train', BASE_IMAGE_PATH))
                
        elif n == 3 or n == 4:
            # 2 to train, 1 to val, 1 to test
            stats['small'] += 1
            for i, item in enumerate(items):
                if i < 2:
                    splits['train'].append(create_output_dict(item, 'train', BASE_IMAGE_PATH))
                elif i == 2:
                    splits['val'].append(create_output_dict(item, 'val', BASE_IMAGE_PATH))
                else:
                    splits['test'].append(create_output_dict(item, 'test', BASE_IMAGE_PATH))
                    
        else:
            # n >= 5: Apply 8:1:1 stratification
            stats['normal'] += 1
            train_cutoff = int(n * 0.8)
            val_cutoff = train_cutoff + int(n * 0.1)
            
            for i, item in enumerate(items):
                if i < train_cutoff:
                    splits['train'].append(create_output_dict(item, 'train', BASE_IMAGE_PATH))
                elif i < val_cutoff:
                    splits['val'].append(create_output_dict(item, 'val', BASE_IMAGE_PATH))
                else:
                    splits['test'].append(create_output_dict(item, 'test', BASE_IMAGE_PATH))
    
    # Print statistics
    print("\n=== Splitting Statistics ===")
    print(f"Groups with 1 sample (all to train): {stats['singleton']}")
    print(f"Groups with 2 samples (all to train): {stats['double']}")
    print(f"Groups with 3-4 samples (2 train, 1 val, 1 test): {stats['small']}")
    print(f"Groups with ≥5 samples (8:1:1 split): {stats['normal']}")
    
    print(f"\n=== Final Split Sizes ===")
    print(f"Training: {len(splits['train'])} samples")
    print(f"Validation: {len(splits['val'])} samples")
    print(f"Test: {len(splits['test'])} samples")
    print(f"Total: {len(splits['train']) + len(splits['val']) + len(splits['test'])} samples")
    
    # Calculate ratios
    total = len(splits['train']) + len(splits['val']) + len(splits['test'])
    print(f"\n=== Split Ratios ===")
    print(f"Train: {len(splits['train'])/total:.1%}")
    print(f"Val: {len(splits['val'])/total:.1%}")
    print(f"Test: {len(splits['test'])/total:.1%}")
    
    # Save output file
    with open(output_file, 'w') as f:
        json.dump(splits, f, indent=2)
    
    print(f"\nSplit data saved to {output_file}")
    
    return splits

# Main execution
if __name__ == "__main__":
    # Update these paths as needed
    INPUT_FILE = "filtered_captions.json"  # Your input JSON file
    OUTPUT_FILE = "train_val_test.json"  # Output file
    
    # Check if input file exists
    if not os.path.exists(INPUT_FILE):
        print(f"Error: Input file '{INPUT_FILE}' not found.")
        print("Please make sure your JSON file is in the same directory as this script.")
        exit(1)
    
    try:
        splits = stratified_split(INPUT_FILE, OUTPUT_FILE)
        print("✅ Dataset splitting completed successfully!")
        
    except Exception as e:
        print(f"❌ Error during splitting: {str(e)}")
        exit(1)


Loaded 8352 samples from filtered_captions.json
Found 848 unique organ-procedure-diagnosis combinations

=== Splitting Statistics ===
Groups with 1 sample (all to train): 357
Groups with 2 samples (all to train): 127
Groups with 3-4 samples (2 train, 1 val, 1 test): 117
Groups with ≥5 samples (8:1:1 split): 247

=== Final Split Sizes ===
Training: 6623 samples
Validation: 727 samples
Test: 1002 samples
Total: 8352 samples

=== Split Ratios ===
Train: 79.3%
Val: 8.7%
Test: 12.0%

Split data saved to train_val_test.json
✅ Dataset splitting completed successfully!


Below code to see the groups and samples based on which the decision was taken

In [2]:
import json
from collections import defaultdict

def extract_info(report):
    """Extract organ, procedure, diagnosis from report"""
    try:
        header, diagnosis = report.split(';\n', 1)
    except ValueError:
        parts = report.split(';')
        if len(parts) > 1:
            header = parts[0]
            diagnosis = ';'.join(parts[1:])
        else:
            header = report
            diagnosis = ""

    header_parts = header.split(',', 1)
    organ = header_parts[0].strip() if len(header_parts) > 0 else ""
    procedure = header_parts[1].strip() if len(header_parts) > 1 else ""
    diagnosis = diagnosis.strip()
    return organ, procedure, diagnosis

def create_groups_breakdown(input_file, output_file):
    """Create detailed breakdown of all groups"""
    
    # Load input data
    with open(input_file, 'r') as f:
        data = json.load(f)
    
    # Group samples by organ-procedure-diagnosis
    groups = defaultdict(list)
    for item in data:
        key = extract_info(item['report'])
        groups[key].append(item['id'])
    
    # Categorize groups by their size
    output = {
        'summary': {
            'total_samples': len(data),
            'total_groups': len(groups),
            'groups_1_sample': 0,
            'groups_2_samples': 0,
            'groups_3_4_samples': 0,
            'groups_5_or_more_samples': 0
        },
        'groups_1_sample': [],
        'groups_2_samples': [],
        'groups_3_4_samples': [],
        'groups_5_or_more_samples': []
    }
    
    # Sort groups by organ name for better readability
    sorted_groups = sorted(groups.items(), key=lambda x: (x[0][0], x[0][1], x[0][2]))
    
    for key, ids in sorted_groups:
        n = len(ids)
        entry = {
            'group': {
                'organ': key[0],
                'procedure': key[1],
                'diagnosis': key[2]
            },
            'sample_count': n,
            'sample_ids': sorted(ids)  # Sort IDs for consistency
        }
        
        if n == 1:
            output['groups_1_sample'].append(entry)
            output['summary']['groups_1_sample'] += 1
        elif n == 2:
            output['groups_2_samples'].append(entry)
            output['summary']['groups_2_samples'] += 1
        elif 3 <= n <= 4:
            output['groups_3_4_samples'].append(entry)
            output['summary']['groups_3_4_samples'] += 1
        else:
            output['groups_5_or_more_samples'].append(entry)
            output['summary']['groups_5_or_more_samples'] += 1
    
    # Save detailed breakdown
    with open(output_file, 'w') as f:
        json.dump(output, f, indent=2)
    
    print(f"✅ Detailed breakdown saved to {output_file}")
    print(f"📊 Summary: {output['summary']}")
    
    return output

# Run the analysis
if __name__ == "__main__":
    INPUT_FILE = "filtered_captions.json"  # Your input file
    OUTPUT_FILE = "grouped_samples_breakdown.json"
    
    breakdown = create_groups_breakdown(INPUT_FILE, OUTPUT_FILE)


✅ Detailed breakdown saved to grouped_samples_breakdown.json
📊 Summary: {'total_samples': 8352, 'total_groups': 848, 'groups_1_sample': 357, 'groups_2_samples': 127, 'groups_3_4_samples': 117, 'groups_5_or_more_samples': 247}


More Human readable txt file below

In [3]:
import json
from collections import defaultdict

def extract_info(report):
    """Extract organ, procedure, diagnosis from report"""
    try:
        header, diagnosis = report.split(';\n', 1)
    except ValueError:
        parts = report.split(';')
        if len(parts) > 1:
            header = parts[0]
            diagnosis = ';'.join(parts[1:])
        else:
            header = report
            diagnosis = ""

    header_parts = header.split(',', 1)
    organ = header_parts[0].strip() if len(header_parts) > 0 else ""
    procedure = header_parts[1].strip() if len(header_parts) > 1 else ""
    diagnosis = diagnosis.strip()
    return organ, procedure, diagnosis

def create_readable_summary(input_file, output_file):
    """Create human-readable text summary"""
    
    # Load input data
    with open(input_file, 'r') as f:
        data = json.load(f)
    
    # Group samples
    groups = defaultdict(list)
    for item in data:
        key = extract_info(item['report'])
        groups[key].append(item['id'])
    
    # Sort groups by organ and size
    sorted_groups = sorted(groups.items(), key=lambda x: (x[0][0], len(x[1]), x[0][1]))
    
    # Create text summary
    with open(output_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("REG2025 DATASET GROUPS BREAKDOWN\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Total Samples: {len(data)}\n")
        f.write(f"Total Unique Groups: {len(groups)}\n\n")
        
        # Categorize and write sections
        categories = {
            'SINGLETON GROUPS (1 sample - all to TRAIN)': [],
            'DOUBLE GROUPS (2 samples - all to TRAIN)': [],
            'SMALL GROUPS (3-4 samples - 2 train, 1 val, 1 test)': [],
            'LARGE GROUPS (5+ samples - 8:1:1 split)': []
        }
        
        for key, ids in sorted_groups:
            n = len(ids)
            group_info = {
                'organ': key[0],
                'procedure': key[1],
                'diagnosis': key[2],
                'count': n,
                'ids': ids
            }
            
            if n == 1:
                categories['SINGLETON GROUPS (1 sample - all to TRAIN)'].append(group_info)
            elif n == 2:
                categories['DOUBLE GROUPS (2 samples - all to TRAIN)'].append(group_info)
            elif 3 <= n <= 4:
                categories['SMALL GROUPS (3-4 samples - 2 train, 1 val, 1 test)'].append(group_info)
            else:
                categories['LARGE GROUPS (5+ samples - 8:1:1 split)'].append(group_info)
        
        # Write each category
        for category_name, groups_list in categories.items():
            f.write(f"\n{category_name}\n")
            f.write("-" * len(category_name) + "\n")
            f.write(f"Count: {len(groups_list)} groups\n\n")
            
            for i, group in enumerate(groups_list, 1):
                f.write(f"{i:3d}. [{group['count']} samples] {group['organ']} | {group['procedure']}\n")
                f.write(f"     Diagnosis: {group['diagnosis'][:100]}{'...' if len(group['diagnosis']) > 100 else ''}\n")
                f.write(f"     Sample IDs: {', '.join(group['ids'][:5])}{'...' if len(group['ids']) > 5 else ''}\n\n")
        
        # Summary statistics
        f.write("\n" + "="*80 + "\n")
        f.write("SUMMARY STATISTICS\n")
        f.write("="*80 + "\n")
        f.write(f"Groups with 1 sample (all to train): {len(categories['SINGLETON GROUPS (1 sample - all to TRAIN)'])}\n")
        f.write(f"Groups with 2 samples (all to train): {len(categories['DOUBLE GROUPS (2 samples - all to TRAIN)'])}\n")
        f.write(f"Groups with 3-4 samples (2 train, 1 val, 1 test): {len(categories['SMALL GROUPS (3-4 samples - 2 train, 1 val, 1 test)'])}\n")
        f.write(f"Groups with ≥5 samples (8:1:1 split): {len(categories['LARGE GROUPS (5+ samples - 8:1:1 split)'])}\n")
    
    print(f"✅ Human-readable summary saved to {output_file}")
    return categories

# Run the analysis
if __name__ == "__main__":
    INPUT_FILE = "filtered_captions.json"  # Your input file
    OUTPUT_FILE = "groups_summary.txt"
    
    summary = create_readable_summary(INPUT_FILE, OUTPUT_FILE)


✅ Human-readable summary saved to groups_summary.txt
